# Discard Policy

After drawing we hold 11 cards and must put one face-up where the opponent can take it. A good discard trades our own deadwood against the risk of feeding opponent melds.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import random
from itertools import combinations

from agent.cards import Card, SUITS, RANKS, RANK_INDEX, make_deck, find_best_melds
from agent.inference import BeliefState

def deadwood(cards):
    """Total point value of a list of (unmelded) cards."""
    return sum(c.value for c in cards)

## 1. The Decision & Score Function

We have 11 cards to choose from. The algorithm chooses the card 'd' which maximises some score function. Define $D(d)$ as the deadwood left after discarding $d$, and R(d) as the risk of $d$ benefiting the opponent. We can compose the function:

$$Score(d)=-(D(d)+\alpha R(d))$$

In [3]:
def melds_containing(d):
    """
    Every concrete 3-card meld d could complete, returned as (partner_a, partner_b) pairs.
    Set melds : d plus two other suits of the same rank  -> C(3,2)=3 combinations.
    Run melds : d as the low / middle / high card of a 3-run in its own suit.
    """
    melds = []
    # Check for sets
    same_rank = [Card(d.rank, s) for s in SUITS if s != d.suit]
    for x, y in combinations(same_rank, 2):
        melds.append((x, y))
    # Check for runs
    ri = RANK_INDEX[d.rank]
    for start in (ri - 2, ri - 1, ri):
        if start < 0 or start + 2 >= len(RANKS):
            continue
        run = [Card(RANKS[start + k], d.suit) for k in range(3)]
        melds.append(tuple(c for c in run if c != d))
    return melds


def keep_term(d, hand):
    """
    D(d): our deadwood points if we discard d (lower is better).
    """
    remaining = [c for c in hand if c != d]
    _, dw = find_best_melds(remaining)
    return deadwood(dw)


def risk_term(d, bs):
    """
    R(d): expected points conceded if the opponent melds d.

    Independence approximation on the BeliefState marginals:
        P(opp holds both partners of a meld) ~= p(x) * p(y)
    weighted by the points that meld is worth, summed over every meld d completes.
    """
    total = 0.0
    for x, y in melds_containing(d):
        total += bs.prob(x) * bs.prob(y) * (d.value + x.value + y.value)
    return total


def discard_score(d, hand, bs, alpha):
    D = keep_term(d, hand)
    R = risk_term(d, bs)
    return -(D + alpha * R), D, R


def best_discard(hand, bs, alpha):
    """The card maximising the score (ties broken by hand order)."""
    return max(hand, key=lambda d: discard_score(d, hand, bs, alpha)[0])

## 2. The Keep Term $D(d)$

Here we calculate $D(d)$, which is the score of our deadwood if we discard $d$. A pure greedy algorithm would look to minimise this function whilst ignoring the opponent completely.

In [10]:
random.seed(1)
deck = make_deck(); random.shuffle(deck)
hand = deck[:11]
face_up = deck[11]
bs = BeliefState(list(hand), face_up)   # all 11 held cards are dead (P=0)

melds, dw = find_best_melds(hand)
print('Hand        :', ' '.join(str(c) for c in hand))
print('Best melds  :', [[str(c) for c in m] for m in melds] or '(none)')
print('Deadwood    :', [str(c) for c in dw], '=', deadwood(dw))
print()
print(f'{"discard":>8}  {"D(d)":>5}')
for d in sorted(hand, key=lambda c: (c.suit, c.rank_idx)):
    print(f'{str(d):>8}  {keep_term(d, hand):>5}')

print('\nGreedy (alpha=0) discards:', best_discard(hand, bs, alpha=0.0))

Hand        : JS 10H QC 10D 3H KC 7D QH 10C 6H 4C
Best melds  : [['10H', '10D', '10C']]
Deadwood    : ['JS', 'QC', '3H', 'KC', '7D', 'QH', '6H', '4C'] = 60

 discard   D(d)
      4C     56
     10C     80
      QC     50
      KC     50
      7D     53
     10D     80
      3H     57
      6H     54
     10H     80
      QH     50
      JS     50

Greedy (alpha=0) discards: JS


## 3. The Risk Term $R(d)$

This is where we take the opponent into account.

Below is a fixed hand being played. The opponent is clearly collecting around J♠, so the risk of discarding it is high. To score that risk we ask: if we hand the opponent $d$, how many points do we expect to concede?

The opponent gains from $d$ only when they already hold the partners that complete a meld with it. Assuming independence — strictly an approximation, since holding $x$ leaves only 9 slots for $y$, so the two are slightly negatively correlated and $p(x\cap y)\le p(x)\,p(y)$ — the chance they hold both partners of a given meld is $p(x)\,p(y)$. Weighting by the points that meld banks for them and summing over every meld $d$ could complete:

$$R(d) \;=\; \sum_{(x,y)\,\in\,\Omega} p(x)\,p(y)\,\big(\text{value}(d)+\text{value}(x)+\text{value}(y)\big)$$

where $\Omega$ is the set of partner pairs $(x,y)$ that form a meld with $d$ (same-rank sets, or runs in $d$'s suit).

Because we add melds rather than combine them as probabilities, $R(d)$ is a risk score, not a calibrated probability.

In [5]:
hand = [Card('J','S'), Card('K','C'),
        Card('3','H'), Card('4','H'), Card('5','H'),
        Card('7','D'), Card('8','D'), Card('9','D'),
        Card('2','C'), Card('2','H'), Card('6','S')]

melds, dw = find_best_melds(hand)
print('Hand      :', ' '.join(str(c) for c in hand))
print('Best melds:', [[str(c) for c in m] for m in melds], '| deadwood', deadwood(dw))

# Opponent visibly collects spades around the jack.
bs = BeliefState(list(hand), Card('A','C'))
bs.observe_opponent_draw_discard(Card('10','S'))   # P(10S) = 1
bs.observe_opponent_draw_discard(Card('Q','S'))    # P(QS)  = 1

print()
print(f'{"discard":>8}  {"D(d)":>5}  {"R(d)":>8}')
for d in [Card('J','S'), Card('K','C')]:
    print(f'{str(d):>8}  {keep_term(d, hand):>5}  {risk_term(d, bs):>8.2f}')
print('\nSame D, very different R: J\u2660 feeds the opponent, K\u2663 is safe.')

Hand      : JS KC 3H 4H 5H 7D 8D 9D 2C 2H 6S
Best melds: [['7D', '8D', '9D'], ['3H', '4H', '5H', '2H']] | deadwood 28

 discard   D(d)      R(d)
      JS     18     50.38
      KC     18      7.50

Same D, very different R: J♠ feeds the opponent, K♣ is safe.


## 4. Risk-aware vs Greedy

We sweep through alpha on the controlled hand to see how changing it changes the preference of the algorithm.

In [6]:
print(f'{"discard":>8}  {"D":>3}  {"R":>7}  {"score(a=0.1)":>12}')
for d in [Card('J','S'), Card('K','C')]:
    s, D, R = discard_score(d, hand, bs, alpha=0.1)
    print(f'{str(d):>8}  {D:>3}  {R:>7.2f}  {s:>12.2f}')

print()
for alpha in [0.0, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]:
    print(f'alpha={alpha:>4}:  discard {best_discard(hand, bs, alpha)}')

 discard    D        R  score(a=0.1)
      JS   18    50.38        -23.04
      KC   18     7.50        -18.75

alpha= 0.0:  discard JS
alpha=0.02:  discard KC
alpha=0.05:  discard KC
alpha= 0.1:  discard KC
alpha= 0.2:  discard KC
alpha= 0.5:  discard KC
alpha= 1.0:  discard KC


## 5. Protecting our melds

$D(d)$ should protect our own melds. We Check (a) at $\alpha=0$ the policy never costs us extra deadwood, and (b) classify any meld-member discard at $\alpha>0$ as either a free drop (redundant card, e.g. the 4th card of a 4-run, $D$ unchanged) or a deliberate safety sacrifice.

In [7]:
random.seed(3)
N = 400
greedy_sacrifice = 0      # alpha=0 chose a higher-deadwood discard than necessary
free_meld_drop   = 0      # alpha=0.3 discarded a meld card but D was unchanged
safety_sacrifice = 0      # alpha=0.3 broke a meld (raised D) to reduce risk

for _ in range(N):
    deck = make_deck(); random.shuffle(deck)
    hand = deck[:11]; face_up = deck[11]
    bs = BeliefState(list(hand), face_up)

    min_D = min(keep_term(c, hand) for c in hand)
    if keep_term(best_discard(hand, bs, 0.0), hand) > min_D:
        greedy_sacrifice += 1

    d = best_discard(hand, bs, 0.3)
    meld_cards = {c for m in find_best_melds(hand)[0] for c in m}
    if d in meld_cards:
        if keep_term(d, hand) <= min_D:
            free_meld_drop += 1
        else:
            safety_sacrifice += 1

print(f'alpha=0   deadwood sacrifices          : {greedy_sacrifice}/{N}')
print(f'alpha=0.3 meld-member, FREE (D same)    : {free_meld_drop}/{N}')
print(f'alpha=0.3 meld-member, SACRIFICED for R : {safety_sacrifice}/{N}')

alpha=0   deadwood sacrifices          : 0/400
alpha=0.3 meld-member, FREE (D same)    : 3/400
alpha=0.3 meld-member, SACRIFICED for R : 0/400


Note: If we're approaching the end of the game, sometimes sacrifices can be beneficial to prevent knocking. So a constant alpha is a simplification and could be adapted.

## Summary

$\alpha$ controls how much weight we put on the opponent's threat. At $\alpha = 0$ the policy is greedy; as $\alpha$ grows the agent trades its own deadwood for safety.

The risk term $R$ uses an independence approximation, $p(x \cap y) \approx p(x)\,p(y)$. Because the hand size is fixed ($\sum_c p(c) = 10$), the partners are in fact slightly negatively correlated, so $p(x)p(y)$ overstates the true joint and therefore overstates the opponent's completion chance. The model is thus conservative — it reads the opponent as more dangerous than they are, erring on the side of caution.

The keep term $D$ protects our own melds for free: across 400 hands the policy never sacrificed deadwood at $\alpha = 0$, and the only meld-member discards at $\alpha > 0$ were redundant cards that cost nothing to drop. No explicit "don't break your melds" rule is needed.